# Datamine COGTRI Process Example

This notebook demonstrates how to configure and run the **`cogtri`** process wrapper in `dmstudio`.

## Process Description

## COGTRI

**Note** : This is a _superprocess_ and running it may have an effect on other Datamine files in the project.

The process calculates the centre point and the dip and dip direction of each triangle in a wireframe.

Output from the process consists of a wireframe (a wireframe triangle and a wireframe points file), and a points file. Both outputs are optional, although at least one must be defined.

The output wireframe points file is a copy of the input wireframe points file. The output wireframe triangle file is a copy of the input wireframe triangle file, but with the following extra fields:

* **XCOG** , **YCOG** , **ZCOG** : The coordinates of the centre of each triangle.
* **XP1** , **YP1** , **ZP1** , **XP2** , **YP2** , **ZP2** , **XP3** , **YP3** , **ZP3** : The coordinates of each vertex of each triangle. These fields are only included if parameter **VERTEX** is set to 1.

The output points file contains the attribute fields of the input wireframe triangle file, plus the following fields:

* **XPT** , **YPT** , **ZPT** : The coordinates of the centre of each triangle.
* **SDIP** , **DIPDIRN** : The dip and dip direction in degrees of each triangle.
* **SYMBOL** , **SYMSIZE** : A symbol code and it's size in mm, as defined by parameters SYMBOL and SYMSIZE.

The default value for SYMBOL is 216, which is a filled arrow. Other SYMBOL codes are shown in the following diagram. The codes for the top line of symbols are 201 to 210, code 211 to 220 for the second row, and so on.

![cogtri-fig2.jpg \(8349 bytes\)](../Images_Commands/cogtri-fig2.jpg)

One of the benefits of including the centre of each triangle in the wireframe triangle file is that a field can be set, for example using the command [EXTRA](<extra.md>), so that the triangles are coloured according to the elevation of each triangle.

The output point data file can be displayed in any **3D** Window. The program automatically recognizes and uses the three numeric fields which control the way in which point data is rendered.

These fields are:

* **SYMSIZE** : the symbol size.
* **DIPDIRN** : the symbol rotation in degrees.
* **SDIP** : the symbol dip in degrees.

The point data file can also be used as input when creating [Stereonet Charts](<../Stereonet/Stereonet%20Introduction.md>).

### Input Files:

* **wtrin** (Wireframe Triangle File):
  Input wireframe triangle file.
  Required=Yes

* **wptin** (Wireframe Points File):
  Input wireframe points file.
  Required=Yes

### Output Files:

* **wtrout** (Wireframe Triangle File):
  Output wireframe triangle file. This contains all the fields from the input wireframe
  triangle file and: - XCOG, YCOG, ZCOG: the XYZ coordinates of the centre of each
  triangle. - if parameter VERTEX is set to 1 then the fields XP1, YP1, ZP1, XP2, YP2,
  ZP2, XP3, YP3, ZP3 representing the vertices of each triangle will also be included.
  Required=No

* **wptout** (Wireframe Points File):
  Output wireframe points file. This is a copy of the input wireframe points file.
  Required=No

* **ptnout** (Point Data File):
  Output point data file containing the following fields: - XPT, YPT, ZPT: the XYZ
  coordinates of the centre of each triangle. - SDIP, DIPDIRN: the dip and dip direction
  of each triangle, in degrees. - SYMBOL, SYMSIZE: the symbol type and symbol size of the
  rotated symbol.
  Required=No

### Fields:

### Parameters:

* **vertex**:
  Flag specifying whether the coordinates of the vertices of each triangle are to be
  included in the output wireframe triangle file WTROUT:
  Range=0,1
  Values=0,1
  Default=0
  Required=No

* **symbol**:
  Code for the rotated symbol to be included in field SYMBOL of the output point data
  file PTNOUT. The default value of 216 is a filled arrow.
  Range=0,400
  Values=Undefined
  Default=216
  Required=No

* **symsize**:
  The size in mm of the rotated symbol.
  Range=0,100
  Values=Undefined
  Default=2
  Required=No

In [ ]:
# Step 1: Connect to Datamine and Verify Active Project
import os
import shutil
import glob
import pandas as pd
from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Verify that the active project matches this folder (case-insensitive) to prevent writing files to the wrong place
active_folder = os.path.normpath(oScript.ActiveProject.Folder).lower()
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()

if active_folder != notebook_folder:
    raise RuntimeError(
        f"Active Datamine Project ({active_folder}) does not match this notebook's folder ({notebook_folder}).\n"
        "Please open the 'Project.rmproj' in this folder inside Datamine Studio RM first!"
    )
print("Active project verified successfully.")

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('cogtri')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine help database locally to this folder using a `t_` prefix. Paths are resolved relatively to ensure portability.

In [ ]:
# Step 3: Initialize example project dataset using relative paths
# Resolve relative path to repository's help database dynamically
repo_root = os.path.abspath(os.path.join(notebook_folder, '..', '..', '..'))
help_db = os.path.join(repo_root, 'datamine_help', 'Database', 'DMTutorials', 'Data', 'VBOP', 'Datamine')

files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

for filename in files_to_copy:
    src = os.path.join(help_db, filename)
    # Strip _vb_ prefix and prepend t_ for local usage
    local_name = "t_" + filename.replace("_vb_", "")
    dst = os.path.join(notebook_folder, local_name)
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f"Initialized dataset: {local_name}")
    else:
        print(f"Warning: Source {filename} not found in help database.")

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute cogtri
print("Running cogtri...")
dm_cmd.cogtri(
    wtrin_i='t_SurfaceTriangles',  # required
    wptin_i='t_input_file',  # required
    # wtrout_o='t_cogtri_out',  # optional
    # wptout_o='t_cogtri_out',  # optional
    # ptnout_o='t_cogtri_out',  # optional
    # vertex_p=0,  # optional
    # symbol_p=216,  # optional
    # symsize_p=2,  # optional
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("cogtri execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = os.path.join(notebook_folder, 't_cogtri_out.dmx')
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob(os.path.join(notebook_folder, "t_*.*"))
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    path = os.path.join(notebook_folder, f)
    if os.path.exists(path):
        try:
            os.remove(path)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = os.path.join(notebook_folder, '__pycache__')
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")